# 🥈 Silver Layer - Validation & Self-Healing

This notebook validates Bronze datasets, applies automated correction rules,
quarantines invalid records, and prepares cleaned datasets for the Gold layer.

In [0]:
import os

from pyspark.sql.functions import *

from pyspark.sql.types import *

from datetime import datetime

from utils.config import *

In [0]:
BUS_BRONZE_PATH = os.path.join(
    BRONZE_PATH,
    "bus_gps"
)

EMERGENCY_BRONZE_PATH = os.path.join(
    BRONZE_PATH,
    "emergency"
)

print(BUS_BRONZE_PATH)
print(EMERGENCY_BRONZE_PATH)

In [0]:
bus_df = spark.read.parquet(
    BUS_BRONZE_PATH
)

emergency_df = spark.read.parquet(
    EMERGENCY_BRONZE_PATH
)

Data Quality Validation

In [0]:
from pyspark.sql.functions import col, current_timestamp


def validate_missing_zone(df):
    return df.filter(col("zone").isNull())

def validate_invalid_gps(df):
    return df.filter(
        (col("latitude") < -90) |
        (col("latitude") > 90) |
        (col("longitude") < -180) |
        (col("longitude") > 180)
    )

def validate_negative_delay(df):
    return df.filter(
        col("delay_minutes") < 0
    )

def validate_future_timestamp(df):
    return df.filter(
        col("timestamp") > current_timestamp()
    )

def validate_duplicate_records(df):

    return df.groupBy(df.columns)\
             .count()\
             .filter("count > 1")

    return df.join(
        duplicate_ids,
        on="bus_id",
        how="inner"
    )

In [0]:
missing_zone_df = validate_missing_zone(bus_df)
invalid_gps_df = validate_invalid_gps(bus_df)
negative_delay_df = validate_negative_delay(bus_df)
future_timestamp_df = validate_future_timestamp(bus_df)
duplicate_bus_df = validate_duplicate_records(bus_df)

In [0]:
print("=" * 50)
print("Validation Summary")
print("=" * 50)
print("Missing Zone      :", missing_zone_df.count())
print("Invalid GPS       :", invalid_gps_df.count())
print("Negative Delay    :", negative_delay_df.count())
print("Future Timestamp  :", future_timestamp_df.count())
print("Duplicate Records :", duplicate_bus_df.count())
print("=" * 50)

# Emergency Data Quality Validation

In [0]:
def validate_missing_zone_emergency(df):
    return df.filter(col("zone").isNull())

def validate_invalid_severity(df):
    return df.filter(
        (col("severity") < 1) |
        (col("severity") > 5)
    )

def validate_negative_response_time(df):
    return df.filter(
        col("response_time") < 0
    )

def validate_future_timestamp_emergency(df):
    return df.filter(
        col("timestamp") > current_timestamp()
    )

def validate_duplicate_records(df):

    return df.groupBy(df.columns) \
             .count() \
             .filter("count > 1")

In [0]:
missing_zone_emergency_df = validate_missing_zone_emergency(emergency_df)
invalid_severity_df = validate_invalid_severity(emergency_df)
negative_response_df = validate_negative_response_time(emergency_df)
future_timestamp_emergency_df = validate_future_timestamp_emergency(emergency_df)
duplicate_emergency_df = validate_duplicate_records(emergency_df)

In [0]:
print("=" * 50)
print("Emergency Validation Summary")
print("=" * 50)
print("Missing Zone             :", missing_zone_emergency_df.count())
print("Invalid Severity         :", invalid_severity_df.count())
print("Negative Response Time   :", negative_response_df.count())
print("Future Timestamp         :", future_timestamp_emergency_df.count())
print("Duplicate Records        :", duplicate_emergency_df.count())
print("=" * 50)

# Bus GPS Self-Healing

In [0]:
from pyspark.sql.functions import lit

# Create working copy from Bronze
bus_silver_df = bus_df

# Initialize repair tracking columns
bus_silver_df = (
    bus_silver_df
    .withColumn("repair_status", lit("clean"))
    .withColumn("repair_reason", lit(None).cast("string"))
    .withColumn("repair_confidence", lit(100))
)

print("✅ Silver dataset initialized.")

## Repair Missing Zone

In [0]:
from pyspark.sql.functions import (
    col,first,when)

In [0]:
# Find the most common zone for each bus

zone_lookup = (
    bus_silver_df
    .filter(col("zone").isNotNull())
    .groupBy("bus_id")
    .agg(first("zone").alias("correct_zone"))
)

zone_lookup.show(5)

In [0]:
# Join lookup table with bus dataset
bus_silver_df = (
    bus_silver_df
    .join(
        zone_lookup,
        on="bus_id",
        how="left"
    )
)
print("✅ Zone lookup joined.")

In [0]:
# Repair missing zones

bus_silver_df = (
    bus_silver_df
    .withColumn(
        "zone",
        when(
            col("zone").isNull(),
            col("correct_zone")
        ).otherwise(col("zone"))
    )
    .withColumn(
        "repair_status",
        when(
            col("correct_zone").isNotNull(),
            "repaired"
        ).otherwise(col("repair_status"))
    )
    .withColumn(
        "repair_reason",
        when(
            col("correct_zone").isNotNull(),
            "Missing Zone"
        ).otherwise(col("repair_reason"))
    )
    .withColumn(
        "repair_confidence",
        when(
            col("correct_zone").isNotNull(),
            95
        ).otherwise(col("repair_confidence"))
    )
)

print("✅ Missing Zone repaired.")

In [0]:
bus_silver_df = bus_silver_df.drop("correct_zone")

In [0]:
# Verify repair

print("Remaining Missing Zones :")

bus_silver_df.filter(
    col("zone").isNull()
).count()

## Self-Healing 2 - Invalid GPS Coordinates

In [0]:
print("=" * 60)
print("Invalid GPS Before Repair")
print("=" * 60)

invalid_gps_before = bus_silver_df.filter(
    (col("latitude") < -90) |
    (col("latitude") > 90) |
    (col("longitude") < -180) |
    (col("longitude") > 180)
).count()

print("Invalid GPS Records :", invalid_gps_before)

In [0]:
bus_silver_df = (
    bus_silver_df
    .withColumn(
        "invalid_gps_flag",
        (
            (col("latitude") < -90) |
            (col("latitude") > 90) |
            (col("longitude") < -180) |
            (col("longitude") > 180)
        )
    )
)
bus_silver_df = (
    bus_silver_df
    .withColumn(
        "latitude",
        when(col("latitude") > 90, 90)
        .when(col("latitude") < -90, -90)
        .otherwise(col("latitude"))
    )
    .withColumn(
        "longitude",
        when(col("longitude") > 180, 180)
        .when(col("longitude") < -180, -180)
        .otherwise(col("longitude"))
    )
)
print("✅ Invalid GPS repaired.")

In [0]:
bus_silver_df = (
    bus_silver_df
    .withColumn(
        "repair_status",
        when(
            col("invalid_gps_flag"),
            "repaired"
        ).otherwise(col("repair_status"))
    )
    .withColumn(
        "repair_reason",
        when(
            col("invalid_gps_flag"),
            "Invalid GPS"
        ).otherwise(col("repair_reason"))
    )
    .withColumn(
        "repair_confidence",
        when(
            col("invalid_gps_flag"),
            100
        ).otherwise(col("repair_confidence"))
    )
)

In [0]:
print("=" * 60)
print("Invalid GPS After Repair")
print("=" * 60)

invalid_gps_after = bus_silver_df.filter(
    (col("latitude") < -90) |
    (col("latitude") > 90) |
    (col("longitude") < -180) |
    (col("longitude") > 180)
).count()

print("Remaining Invalid GPS :", invalid_gps_after)

In [0]:
bus_silver_df = bus_silver_df.drop("invalid_gps_flag")

## Self-Healing 3 - Negative Delay

In [0]:
print("=" * 60)
print("Negative Delay Before Repair")
print("=" * 60)
negative_before = bus_silver_df.filter(
    col("delay_minutes") < 0
).count()
print("Negative Delay :", negative_before)

In [0]:
bus_silver_df = (
    bus_silver_df
    .withColumn(
        "negative_delay_flag",
        col("delay_minutes") < 0
    )
)

In [0]:
bus_silver_df = (
    bus_silver_df
    .withColumn(
        "delay_minutes",
        when(
            col("delay_minutes") < 0,
            0
        ).otherwise(col("delay_minutes"))
    )
    .withColumn(
        "repair_status",
        when(
            col("negative_delay_flag"),
            "repaired"
        ).otherwise(col("repair_status"))
    )
    .withColumn(
        "repair_reason",
        when(
            col("negative_delay_flag"),
            "Negative Delay"
        ).otherwise(col("repair_reason"))
    )
    .withColumn(
        "repair_confidence",
        when(
            col("negative_delay_flag"),
            100
        ).otherwise(col("repair_confidence"))
    )
)

print("✅ Negative Delay repaired.")

In [0]:
print("=" * 60)
print("Negative Delay After Repair")
print("=" * 60)

print(
    bus_silver_df.filter(
        col("delay_minutes") < 0
    ).count()
)

In [0]:
bus_silver_df = bus_silver_df.drop("negative_delay_flag")

## Self-Healing 4 - Future Timestamp

In [0]:
print("=" * 60)
print("Future Timestamp Before Repair")
print("=" * 60)

future_before = bus_silver_df.filter(
    col("timestamp") > current_timestamp()
).count()

print("Future Timestamp :", future_before)

In [0]:
bus_silver_df = (
    bus_silver_df
    .withColumn(
        "future_timestamp_flag",
        col("timestamp") > current_timestamp()
    )
)

In [0]:
bus_silver_df = (
    bus_silver_df
    .withColumn(
        "timestamp",
        when(
            col("future_timestamp_flag"),
            current_timestamp()
        ).otherwise(col("timestamp"))
    )
    .withColumn(
        "repair_status",
        when(
            col("future_timestamp_flag"),
            "repaired"
        ).otherwise(col("repair_status"))
    )
    .withColumn(
        "repair_reason",
        when(
            col("future_timestamp_flag"),
            "Future Timestamp"
        ).otherwise(col("repair_reason"))
    )
    .withColumn(
        "repair_confidence",
        when(
            col("future_timestamp_flag"),
            100
        ).otherwise(col("repair_confidence"))
    )
)

print("✅ Future Timestamp repaired.")

In [0]:
print("=" * 60)
print("Future Timestamp After Repair")
print("=" * 60)

print(
    bus_silver_df.filter(
        col("timestamp") > current_timestamp()
    ).count()
)

In [0]:
bus_silver_df = bus_silver_df.drop("future_timestamp_flag")

## Self-Healing 5 - Duplicate Records

In [0]:
print("=" * 60)
print("Duplicate Records Before Repair")
print("=" * 60)

duplicate_before = (
    bus_silver_df.count()
    - bus_silver_df.dropDuplicates().count()
)

print("Duplicate Records :", duplicate_before)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy(
    "bus_id",
    "route_id",
    "zone",
    "latitude",
    "longitude",
    "speed_kmh",
    "delay_minutes",
    "occupancy",
    "timestamp"
).orderBy("bus_id")

bus_silver_df = (
    bus_silver_df
    .withColumn(
        "duplicate_flag",
        row_number().over(window_spec)
    )
)

print("✅ Exact duplicate records identified.")

In [0]:
duplicate_removed = (
    bus_silver_df
    .filter(col("duplicate_flag") > 1)
    .count()
)
bus_silver_df = (
    bus_silver_df
    .filter(col("duplicate_flag") == 1)
    .drop("duplicate_flag")
)
print("✅ Exact duplicate records removed.")

In [0]:
print("=" * 60)
print("Duplicate Records After Repair")
print("=" * 60)

duplicate_after = (
    bus_silver_df.count()
    - bus_silver_df.dropDuplicates().count()
)

print("Remaining Duplicates :", duplicate_after)
print("Removed :", duplicate_removed)

In [0]:
print("=" * 60)
print("Duplicate Records After Repair")
print("=" * 60)
duplicate_after = (
    bus_silver_df.count()
    - bus_silver_df.dropDuplicates().count()
)
print("Remaining Duplicates :", duplicate_after)
print("Removed :", duplicate_removed)

# Emergency Self-Healing

## Initialize Emergency Silver Dataset

In [0]:
emergency_silver_df = emergency_df

emergency_silver_df = (
    emergency_silver_df
    .withColumn("repair_status", lit("clean"))
    .withColumn("repair_reason", lit(None).cast("string"))
    .withColumn("repair_confidence", lit(None).cast("int"))
)

print("✅ Emergency repair tracking initialized.")

## Self-Healing 1 - Missing Zone

In [0]:
from pyspark.sql.functions import col, first, when

In [0]:
zone_lookup = (
    emergency_silver_df
    .filter(col("zone").isNotNull())
    .groupBy("incident_id")
    .agg(first("zone").alias("correct_zone"))
)

In [0]:
emergency_silver_df = (
    emergency_silver_df
    .join(zone_lookup, on="incident_id", how="left")
)

In [0]:
emergency_silver_df = (
    emergency_silver_df
    .withColumn(
        "zone",
        when(
            col("zone").isNull(),
            col("correct_zone")
        ).otherwise(col("zone"))
    )
    .withColumn(
        "repair_status",
        when(
            col("correct_zone").isNotNull(),
            "repaired"
        ).otherwise(col("repair_status"))
    )
    .withColumn(
        "repair_reason",
        when(
            col("correct_zone").isNotNull(),
            "Missing Zone"
        ).otherwise(col("repair_reason"))
    )
    .withColumn(
        "repair_confidence",
        when(
            col("correct_zone").isNotNull(),
            95
        ).otherwise(col("repair_confidence"))
    )
)

In [0]:
emergency_silver_df = emergency_silver_df.drop("correct_zone")

In [0]:
print(
    emergency_silver_df.filter(
        col("zone").isNull()
    ).count()
)

In [0]:
from pyspark.sql.functions import col, when, lit

emergency_silver_df = (
    emergency_silver_df
    .withColumn(
        "zone",
        when(
            col("zone").isNull(),
            "Unknown"
        ).otherwise(col("zone"))
    )
    .withColumn(
        "repair_status",
        when(
            (col("zone") == "Unknown") &
            (col("repair_status") == "clean"),
            "quarantined"
        ).otherwise(col("repair_status"))
    )
    .withColumn(
        "repair_reason",
        when(
            (col("zone") == "Unknown") &
            (col("repair_reason").isNull()),
            "Missing Zone"
        ).otherwise(col("repair_reason"))
    )
    .withColumn(
        "repair_confidence",
        when(
            (col("zone") == "Unknown") &
            (col("repair_confidence").isNull()),
            0
        ).otherwise(col("repair_confidence"))
    )
)

print("✅ Remaining missing zones marked as Unknown.")

In [0]:
print("=" * 60)
print("Emergency Missing Zone Verification")
print("=" * 60)

print(
    emergency_silver_df.filter(
        col("zone").isNull()
    ).count()
)

print(
    emergency_silver_df.filter(
        col("zone") == "Unknown"
    ).count()
)

## Self-Healing 2 - Invalid Severity

In [0]:
print("=" * 60)
print("Invalid Severity Before Repair")
print("=" * 60)

print(
    emergency_silver_df.filter(
        (col("severity") < 1) |
        (col("severity") > 5)
    ).count()
)

In [0]:
emergency_silver_df = (
    emergency_silver_df
    .withColumn(
        "invalid_severity_flag",
        (col("severity") < 1) |
        (col("severity") > 5)
    )
)

In [0]:
emergency_silver_df = (
    emergency_silver_df
    .withColumn(
        "severity",
        when(col("severity") < 1, 1)
        .when(col("severity") > 5, 5)
        .otherwise(col("severity"))
    )
    .withColumn(
        "repair_status",
        when(
            col("invalid_severity_flag"),
            "repaired"
        ).otherwise(col("repair_status"))
    )
    .withColumn(
        "repair_reason",
        when(
            col("invalid_severity_flag"),
            "Invalid Severity"
        ).otherwise(col("repair_reason"))
    )
    .withColumn(
        "repair_confidence",
        when(
            col("invalid_severity_flag"),
            100
        ).otherwise(col("repair_confidence"))
    )
)

print("✅ Invalid Severity repaired.")

In [0]:
print("=" * 60)
print("Invalid Severity After Repair")
print("=" * 60)

print(
    emergency_silver_df.filter(
        (col("severity") < 1) |
        (col("severity") > 5)
    ).count()
)

In [0]:
emergency_silver_df = emergency_silver_df.drop("invalid_severity_flag")

## Self-Healing 3 - Negative Response Time

In [0]:
print("=" * 60)
print("Negative Response Time Before Repair")
print("=" * 60)

print(
    emergency_silver_df.filter(
        col("response_time") < 0
    ).count()
)

In [0]:
emergency_silver_df = (
    emergency_silver_df
    .withColumn(
        "negative_response_flag",
        col("response_time") < 0
    )
)

In [0]:
emergency_silver_df = (
    emergency_silver_df
    .withColumn(
        "response_time",
        when(
            col("response_time") < 0,
            0
        ).otherwise(col("response_time"))
    )
    .withColumn(
        "repair_status",
        when(
            col("negative_response_flag"),
            "repaired"
        ).otherwise(col("repair_status"))
    )
    .withColumn(
        "repair_reason",
        when(
            col("negative_response_flag"),
            "Negative Response Time"
        ).otherwise(col("repair_reason"))
    )
    .withColumn(
        "repair_confidence",
        when(
            col("negative_response_flag"),
            100
        ).otherwise(col("repair_confidence"))
    )
)

print("✅ Negative Response Time repaired.")

In [0]:
print("=" * 60)
print("Negative Response Time After Repair")
print("=" * 60)

print(
    emergency_silver_df.filter(
        col("response_time") < 0
    ).count()
)

In [0]:
emergency_silver_df = emergency_silver_df.drop("negative_response_flag")

## Self-Healing 4 - Future Timestamp

In [0]:
print("=" * 60)
print("Future Timestamp Before Repair")
print("=" * 60)
print(
    emergency_silver_df.filter(
        col("timestamp") > current_timestamp()
    ).count()
)

In [0]:
emergency_silver_df = (
    emergency_silver_df
    .withColumn(
        "future_timestamp_flag",
        col("timestamp") > current_timestamp()
    )
)

In [0]:
emergency_silver_df = (
    emergency_silver_df
    .withColumn(
        "timestamp",
        when(
            col("future_timestamp_flag"),
            current_timestamp()
        ).otherwise(col("timestamp"))
    )
    .withColumn(
        "repair_status",
        when(
            col("future_timestamp_flag"),
            "repaired"
        ).otherwise(col("repair_status"))
    )
    .withColumn(
        "repair_reason",
        when(
            col("future_timestamp_flag"),
            "Future Timestamp"
        ).otherwise(col("repair_reason"))
    )
    .withColumn(
        "repair_confidence",
        when(
            col("future_timestamp_flag"),
            100
        ).otherwise(col("repair_confidence"))
    )
)
print("✅ Future Timestamp repaired.")

In [0]:
print("=" * 60)
print("Future Timestamp After Repair")
print("=" * 60)
print(
    emergency_silver_df.filter(
        col("timestamp") > current_timestamp()
    ).count()
)

In [0]:
emergency_silver_df = emergency_silver_df.drop("future_timestamp_flag")

## Self-Healing 5 - Duplicate Records

In [0]:
print("=" * 60)
print("Duplicate Records Before Repair")
print("=" * 60)

duplicate_before = (
    emergency_silver_df.count()
    - emergency_silver_df.dropDuplicates().count()
)

print("Duplicate Records :", duplicate_before)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy(
    "incident_id",
    "zone",
    "incident_type",
    "severity",
    "response_time",
    "status",
    "timestamp"
).orderBy("incident_id")
emergency_silver_df = (
    emergency_silver_df
    .withColumn(
        "duplicate_flag",
        row_number().over(window_spec)
    )
)
print("✅ Duplicate records identified.")

In [0]:
duplicate_removed = (
    emergency_silver_df
    .filter(col("duplicate_flag") > 1)
    .count()
)
emergency_silver_df = (
    emergency_silver_df
    .filter(col("duplicate_flag") == 1)
    .drop("duplicate_flag")
)
print("✅ Duplicate records removed.")

In [0]:
print("=" * 60)
print("Duplicate Records After Repair")
print("=" * 60)
duplicate_after = (
    emergency_silver_df.count()
    - emergency_silver_df.dropDuplicates().count()
)
print("Remaining Duplicates :", duplicate_after)
print("Removed :", duplicate_removed)

# Quarantine Dataset

In [0]:
bus_quarantine_df = bus_silver_df.filter(
    (col("repair_status") == "quarantined") |
    (col("repair_confidence") < 90)
)
print("Bus Quarantine Records :", bus_quarantine_df.count())

In [0]:
emergency_quarantine_df = emergency_silver_df.filter(
    (col("repair_status") == "quarantined") |
    (col("repair_confidence") < 90)
)
print("Emergency Quarantine Records :", emergency_quarantine_df.count())

# Separate Silver and Quarantine

In [0]:
bus_silver_df = bus_silver_df.filter(
    col("repair_status") != "quarantined"
)
print("Final Bus Silver Records :", bus_silver_df.count())

In [0]:
emergency_silver_df = emergency_silver_df.filter(
    col("repair_status") != "quarantined"
)
print("Final Emergency Silver Records :", emergency_silver_df.count())

# Quarantine Metadata

In [0]:
from pyspark.sql.functions import current_timestamp, lit
bus_quarantine_df = (
    bus_quarantine_df
    .withColumn(
        "quarantine_reason",
        lit("Unable to repair confidently")
    )
    .withColumn(
        "quarantine_timestamp",
        current_timestamp()
    )
)
print("✅ Bus quarantine metadata added.")

In [0]:
emergency_quarantine_df = (
    emergency_quarantine_df
    .withColumn(
        "quarantine_reason",
        lit("Unable to repair confidently")
    )
    .withColumn(
        "quarantine_timestamp",
        current_timestamp()
    )
)

print("✅ Emergency quarantine metadata added.")

# Repair Summary

In [0]:
print("=" * 70)
print("BUS GPS SILVER SUMMARY")
print("=" * 70)

print("Silver Records      :", bus_silver_df.count())
print("Quarantine Records  :", bus_quarantine_df.count())

print("\n")

print("=" * 70)
print("EMERGENCY SILVER SUMMARY")
print("=" * 70)

print("Silver Records      :", emergency_silver_df.count())
print("Quarantine Records  :", emergency_quarantine_df.count())

# Create Silver Batch Folder

In [0]:
from datetime import datetime
BATCH_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
print("Silver Batch ID :", BATCH_ID)

In [0]:
import os

SILVER_BATCH_PATH = os.path.join(
    SILVER_PATH,
    BATCH_ID
)

os.makedirs(SILVER_BATCH_PATH, exist_ok=True)

print("Silver Batch Folder:")
print(SILVER_BATCH_PATH)

# Save Silver Datasets

In [0]:
BUS_SILVER_PATH = os.path.join(
    SILVER_BATCH_PATH,
    "bus_gps"
)

EMERGENCY_SILVER_PATH = os.path.join(
    SILVER_BATCH_PATH,
    "emergency"
)

In [0]:
bus_silver_df.write.mode("overwrite").parquet(
    BUS_SILVER_PATH
)
print("✅ Bus Silver saved.")

In [0]:
emergency_silver_df.write.mode("overwrite").parquet(
    EMERGENCY_SILVER_PATH
)
print("✅ Emergency Silver saved.")

# Save Quarantine Datasets

In [0]:
BUS_QUARANTINE_PATH = os.path.join(
    SILVER_BATCH_PATH,
    "bus_quarantine"
)
EMERGENCY_QUARANTINE_PATH = os.path.join(
    SILVER_BATCH_PATH,
    "emergency_quarantine"
)

In [0]:
bus_quarantine_df.write.mode("overwrite").parquet(
    BUS_QUARANTINE_PATH
)
print("✅ Bus Quarantine saved.")

In [0]:
emergency_quarantine_df.write.mode("overwrite").parquet(
    EMERGENCY_QUARANTINE_PATH
)
print("✅ Emergency Quarantine saved.")

# Verify Silver Output

In [0]:
print("=" * 70)
print("Silver Folder Contents")
print("=" * 70)
for folder in os.listdir(SILVER_BATCH_PATH):
    print("📁", folder)